<a href="https://colab.research.google.com/github/Pedro-Lucas-Vieira/Aurora-AI/blob/main/CHAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# CHATBOT DA AURORA S.A.
# ============================================================

import os

from dotenv import load_dotenv

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from langchain_community.vectorstores import FAISS


# ============================================================
# CONFIGURAÇÕES
# ============================================================

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY não encontrada no .env")


# ============================================================
# EMBEDDINGS
# ============================================================

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    google_api_key=GOOGLE_API_KEY
)


# ============================================================
# CARREGA A BASE FAISS
# ============================================================

print("Carregando base vetorial...")

vectorstore = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization=True
)

print("Base vetorial carregada com sucesso.")


# ============================================================
# MODELO GEMINI
# ============================================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2
)


# ============================================================
# HISTÓRICO
# ============================================================

historico = []


# ============================================================
# FUNÇÃO PRINCIPAL
# ============================================================

def responder(pergunta):

    # Pesquisa na base de documentos
    documentos = vectorstore.similarity_search(
        pergunta,
        k=4
    )

    # Junta os documentos encontrados
    contexto = ""

    for documento in documentos:
        contexto += documento.page_content
        contexto += "\n\n"

    # Pega as últimas mensagens
    conversa = "\n".join(historico[-8:])

    # Prompt
    prompt = f"""
Você é o AURORA AI, assistente interno da AURORA S.A.

Responda utilizando SOMENTE as informações presentes
na documentação abaixo.

Não invente informações.

Se a resposta não estiver na documentação, responda:

"Não encontrei essa informação na documentação da empresa."

Use uma linguagem profissional, clara e objetiva.

Não mencione nomes de arquivos ou detalhes técnicos.

Se a pergunta não estiver relacionada à documentação
da empresa, responda:

"Posso ajudar apenas com informações presentes na
documentação interna da AURORA S.A."

HISTÓRICO:
{conversa}

DOCUMENTAÇÃO:
{contexto}

PERGUNTA:
{pergunta}
"""

    # Envia para o Gemini
    resposta = llm.invoke(prompt)

    # A resposta pode vir como texto ou lista
    if isinstance(resposta.content, str):

        texto = resposta.content

    else:

        texto = ""

        for item in resposta.content:

            if isinstance(item, dict) and "text" in item:
                texto += item["text"]

            elif isinstance(item, str):
                texto += item

    # Salva no histórico
    historico.append(
        f"Usuário: {pergunta}"
    )

    historico.append(
        f"Assistente: {texto}"
    )

    # Mantém somente as últimas mensagens
    if len(historico) > 20:
        historico.pop(0)
        historico.pop(0)

 return texto.strip()